<a href="https://colab.research.google.com/github/peperjet/bc-ml/blob/main/flight_delay/flight_delay_260401.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

항공편 지연 예측 AI

In [ ]:
import numpy as np
import pandas as pd
import random #랜덤값 제어
import os #랜덤 고정
import warnings # warnings 모듈 추가

# 2. 데이터 처리
import numpy as np
import pandas as pd

# 3. 전처리 / 검증
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

# 4. 모델
import lightgbm as lgb

# 5. 경고 숨기기
warnings.filterwarnings('ignore')

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(42)

In [ ]:
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')
sample_submission = pd.read_csv('/content/sample_submission.csv')

print(train.shape, test.shape, sample_submission.shape)
train.head()

In [ ]:
# 타겟 확인
print(train.columns)

In [ ]:
# 타겟 숫자로 변환

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
train['Delay'] = le.fit_transform(train['Delay'])

In [ ]:
# 결측치 확인
train.isnull().sum()

결측치 있는 컬럼

Estimated_Departure_Time
Estimated_Arrival_Time
Origin_State
Destination_State
Airline
Carrier_Code(IATA)
Carrier_ID(DOT)

In [ ]:
# 최빈값으로 채우기

nan_cols = [
    'Estimated_Departure_Time',
    'Estimated_Arrival_Time',
    'Origin_State',
    'Destination_State',
    'Airline',
    'Carrier_Code(IATA)',
    'Carrier_ID(DOT)'
]

for col in nan_cols:
    mode_value = train[col].mode()[0]
    train[col] = train[col].fillna(mode_value)
    test[col] = test[col].fillna(mode_value)

print(train[nan_cols].isnull().sum())
print(test[nan_cols].isnull().sum())

In [ ]:
# delay 숫자로 바꾸기

from sklearn.preprocessing import LabelEncoder

target_le = LabelEncoder()
train['Delay'] = target_le.fit_transform(train['Delay'])

print(train['Delay'].value_counts())
print(target_le.classes_)

In [ ]:
# 범주형 컬럼 찾기

cat_cols = [
    'Origin_Airport',
    'Origin_State',
    'Destination_Airport',
    'Destination_State',
    'Airline',
    'Carrier_Code(IATA)',
    'Tail_Number'
]

In [ ]:
# 범주형 인코딩

for col in cat_cols:
    le = LabelEncoder()
    le.fit(pd.concat([train[col], test[col]], axis=0).astype(str))

    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

print('범주형 인코딩 완료')

In [ ]:
# 피처 / 타겟 ㅈ나누기

X = train.drop(columns=['ID', 'Delay'])
y = train['Delay']
X_test = test.drop(columns=['ID'])

print(X.shape, y.shape, X_test.shape)

In [ ]:
print(X.isnull().sum().sum())
print(X_test.isnull().sum().sum())

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros((len(X), len(target_le.classes_)))
test_pred = np.zeros((len(X_test), len(target_le.classes_)))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(
        objective='multiclass',
        num_class=len(target_le.classes_),
        n_estimators=300,
        learning_rate=0.05,
        random_state=42
    )

    model.fit(X_train, y_train)

    val_pred = model.predict_proba(X_val)
    oof_pred[val_idx] = val_pred
    test_pred += model.predict_proba(X_test) / skf.n_splits

    fold_logloss = log_loss(y_val, val_pred)
    print(f'Fold {fold+1} LogLoss: {fold_logloss:.5f}')

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.476384 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2024
[LightGBM] [Info] Number of data points in the train set: 800000, number of used features: 15
[LightGBM] [Info] Start training from score -3.101093
[LightGBM] [Info] Start training from score -1.560642
[LightGBM] [Info] Start training from score -0.294373
Fold 1 LogLoss: 0.67999
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.037931 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2033
[LightGBM] [Info] Number of data points in the train set: 800000, number of used features: 15
[LightGBM] [Info] Start training from score -3.101093
[LightGBM] [Info] Start training from score -1.560642
[LightGBM] [Info] Start training from score -0.294373


In [ ]:
# 전체 cv 점수 확인

cv_score = log_loss(y, oof_pred)
print('CV LogLoss:', cv_score)